In [ ]:
from google.colab import drive
import os

MOUNT_POINT = '/content/drive'
if os.path.exists(f'{MOUNT_POINT}/MyDrive'):
    print('Drive already mounted')
else:
    try:
        drive.mount(MOUNT_POINT)
    except ValueError as e:
        if 'Mountpoint must not already contain files' not in str(e):
            raise
        MOUNT_POINT = '/content/gdrive'
        drive.mount(MOUNT_POINT)

DRIVE_ROOT = f'{MOUNT_POINT}/MyDrive'
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# ==============================================================
# CELL 1b: RECOVERY - Khoi phuc checkpoint + sua best_model
# ==============================================================
import os, shutil, torch

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive')
DRIVE_PROJECT = f'{DRIVE_ROOT}/tiki_absa'
COLAB_PROJECT = '/content/tiki'

def restore_and_clean(drive_ckpt, local_ckpt):
    """
    Copy checkpoint tu Drive ve local, kiem tra tung file,
    xoa file corrupt o CA HAI noi (local + Drive).
    Tra ve danh sach epoch hop le.
    """
    if not os.path.exists(drive_ckpt):
        return []

    os.makedirs(local_ckpt, exist_ok=True)
    shutil.copytree(drive_ckpt, local_ckpt, dirs_exist_ok=True)

    pts = [f for f in os.listdir(local_ckpt)
           if f.startswith('epoch_') and f.endswith('.pt')]
    good, bad = [], []

    for fname in pts:
        local_path = os.path.join(local_ckpt, fname)
        try:
            torch.load(local_path, map_location='cpu')
            good.append(fname)
        except Exception as e:
            bad.append(fname)
            # Xoa ca local lan Drive
            os.remove(local_path)
            drive_path = os.path.join(drive_ckpt, fname)
            if os.path.exists(drive_path):
                os.remove(drive_path)
            print(f'  [X] {fname} CORRUPT ({type(e).__name__}) - da xoa local + Drive')

    if good:
        print(f'  [OK] Checkpoints hop le: {sorted(good)}')
    return good


def try_load_best(ckpt_dir):
    """Load checkpoint tot nhat (epoch lon nhat con hop le)."""
    if not os.path.exists(ckpt_dir):
        return None, 0
    pts = [f for f in os.listdir(ckpt_dir)
           if f.startswith('epoch_') and f.endswith('.pt')]
    if not pts:
        return None, 0
    epochs = sorted([int(p.replace('epoch_','').replace('.pt','')) for p in pts], reverse=True)
    for ep in epochs:
        path = os.path.join(ckpt_dir, f'epoch_{ep}.pt')
        try:
            state = torch.load(path, map_location='cpu')
            return state, ep
        except Exception:
            os.remove(path)
    return None, 0


# ── Buoc 1: Restore + clean checkpoints ──────────────────────
print('=' * 55)
print('  RECOVERY: Restore va kiem tra checkpoint')
print('=' * 55)

for model_name in ['phobert', 'vit5', 'bilstm']:
    print(f'\n[{model_name}]')
    drive_ckpt  = f'{DRIVE_PROJECT}/checkpoints/{model_name}'
    local_ckpt  = f'{COLAB_PROJECT}/checkpoints/{model_name}'
    drive_model = f'{DRIVE_PROJECT}/models/{model_name}'
    local_model = f'{COLAB_PROJECT}/models/{model_name}'

    good = restore_and_clean(drive_ckpt, local_ckpt)
    if not good:
        print(f'  Khong co checkpoint hop le tren Drive')

    # Restore model dir (best_model / epoch dirs)
    if os.path.exists(drive_model):
        os.makedirs(local_model, exist_ok=True)
        shutil.copytree(drive_model, local_model, dirs_exist_ok=True)
        print(f'  models/{model_name}/ restored: {os.listdir(local_model)}')


# ── Buoc 2: Tao lai best_model neu thieu ──────────────────────
print('\n--- Kiem tra best_model ---')
for model_name in ['phobert', 'bilstm']:
    model_dir = f'{COLAB_PROJECT}/models/{model_name}'
    ckpt_dir  = f'{COLAB_PROJECT}/checkpoints/{model_name}'
    best_path = f'{model_dir}/best_model.pt'

    if os.path.exists(best_path):
        try:
            ckpt  = torch.load(best_path, map_location='cpu')
            val_m = ckpt.get('val_metrics', {})
            ep    = ckpt.get('epoch', '?')
            print(f'[{model_name}] best_model.pt OK '
                  f'(epoch {ep}, avg_f1={val_m.get("avg_f1", 0):.4f})')
            continue
        except Exception as e:
            print(f'[{model_name}] best_model.pt CORRUPT ({type(e).__name__}), tao lai...')
            os.remove(best_path)

    # Khong co hoac corrupt -> tao lai tu checkpoint tot nhat
    print(f'[{model_name}] best_model.pt khong ton tai -> tim checkpoint...')
    state, ep = try_load_best(ckpt_dir)
    if state is None:
        print(f'  -> Khong co checkpoint hop le. Can train lai tu dau.')
        continue
    os.makedirs(model_dir, exist_ok=True)
    torch.save({
        'epoch':       ep,
        'model_state': state['model_state'],
        'val_metrics': state.get('val_metrics', {}),
    }, best_path)
    val_m = state.get('val_metrics', {})
    print(f'  -> Tao lai best_model.pt tu epoch {ep} '
          f'(avg_f1={val_m.get("avg_f1", 0):.4f})')
    print(f'     => Cell train se resume tu epoch {ep + 1}')

    # Sync best_model moi len Drive
    drive_model = f'{DRIVE_PROJECT}/models/{model_name}'
    os.makedirs(drive_model, exist_ok=True)
    shutil.copy2(best_path, os.path.join(drive_model, 'best_model.pt'))
    print(f'  -> Da sync best_model.pt len Drive')

# ── Buoc 3: Vit5 ──────────────────────────────────────────────
vit5_ckpt_dir = f'{COLAB_PROJECT}/checkpoints/vit5'
print()
_, ep = try_load_best(vit5_ckpt_dir)
if ep > 0:
    print(f'[vit5] Checkpoint hop le moi nhat: epoch {ep} -> resume tu epoch {ep + 1}')
else:
    print('[vit5] Khong co checkpoint hop le')

print()
print('XONG. Chay Cell 6/7/8 de tiep tuc train.')

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 2: Setup thư mục dự án
# ══════════════════════════════════════════════════════
import os, shutil

# ─── CẤU HÌNH: thay đổi nếu cần ───────────────────────
DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive')
DRIVE_PROJECT = f'{DRIVE_ROOT}/tiki_absa'  # thư mục trên Drive
COLAB_PROJECT = '/content/tiki'                      # thư mục làm việc
# ────────────────────────────────────────────────────────

os.makedirs(COLAB_PROJECT, exist_ok=True)

# Copy toàn bộ project từ Drive về Colab workspace
if os.path.exists(DRIVE_PROJECT):
    print(f'Copying {DRIVE_PROJECT} → {COLAB_PROJECT}...')
    # Copy src
    if os.path.exists(f'{DRIVE_PROJECT}/src'):
        shutil.copytree(f'{DRIVE_PROJECT}/src', f'{COLAB_PROJECT}/src',
                        dirs_exist_ok=True)
    # Copy data
    if os.path.exists(f'{DRIVE_PROJECT}/data'):
        shutil.copytree(f'{DRIVE_PROJECT}/data', f'{COLAB_PROJECT}/data',
                        dirs_exist_ok=True)
    # Copy requirements
    for f in ['requirements_new.txt', 'requirements.txt']:
        src = f'{DRIVE_PROJECT}/{f}'
        if os.path.exists(src):
            shutil.copy2(src, f'{COLAB_PROJECT}/{f}')
    # Restore checkpoints + models (nếu có từ lần train trước)
    for folder in ['checkpoints', 'models', 'results']:
        src = f'{DRIVE_PROJECT}/{folder}'
        if os.path.exists(src):
            shutil.copytree(src, f'{COLAB_PROJECT}/{folder}', dirs_exist_ok=True)
            print(f'  ✅ Restored {folder}/')
    print('✅ Copy hoàn tất!')
else:
    print(f'⚠️  {DRIVE_PROJECT} chưa tồn tại trên Drive.')
    print('   Hãy upload project lên Drive trước (xem hướng dẫn bên dưới).')

# Tạo tất cả thư mục cần thiết
dirs_needed = [
    f'{COLAB_PROJECT}/data/processed',
    f'{COLAB_PROJECT}/data/training',
    f'{COLAB_PROJECT}/data/raw',
    f'{COLAB_PROJECT}/checkpoints/bilstm',
    f'{COLAB_PROJECT}/checkpoints/phobert',
    f'{COLAB_PROJECT}/checkpoints/vit5',
    f'{COLAB_PROJECT}/models/bilstm',
    f'{COLAB_PROJECT}/models/phobert',
    f'{COLAB_PROJECT}/models/vit5',
    f'{COLAB_PROJECT}/results',
    f'{COLAB_PROJECT}/src/training',
]
for d in dirs_needed:
    os.makedirs(d, exist_ok=True)

# Set working directory
os.chdir(COLAB_PROJECT)
import sys
sys.path.insert(0, COLAB_PROJECT)

print(f'\n✅ Working directory: {os.getcwd()}')
print('\nCấu trúc thư mục:')
for d in ['src/training', 'data/processed', 'data/training',
          'checkpoints', 'models', 'results']:
    exists = '✅' if os.path.exists(d) else '❌'
    print(f'  {exists} {d}/')

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 3: Cai dependencies
# Phien ban cu the de tuong thich voi VietAI/vit5-base
# ══════════════════════════════════════════════════════
import subprocess, sys

packages = [
    # transformers==4.35.2: phien ban cuoi cung ho tro VietAI/vit5-base
    # Cac phien ban moi hon (>=4.36) bi loi KeyError:0 va Unigram dict
    "transformers==4.35.2",
    "sentencepiece==0.1.99",   # phai dung voi transformers==4.35.2
    "tokenizers==0.15.2",      # tuong thich voi transformers==4.35.2
    "torch>=2.1.0",
    "torchcrf>=1.1.0",
    "scikit-learn>=1.3.0",
    "numpy>=1.24.0",
    "pandas>=2.0.0",
    "accelerate>=0.26.0",
]

print("Dang cai packages (co the mat 1-2 phut)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Cai dat thanh cong!")
else:
    print("Co loi:", result.stderr[-300:])

import torch, transformers
print(f"PyTorch       : {torch.__version__}")
print(f"Transformers  : {transformers.__version__}")
print(f"CUDA          : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 4: Kiểm tra dữ liệu
# ══════════════════════════════════════════════════════
import json
from collections import Counter

data_file = 'data/processed/asqp_annotated.jsonl'

if not os.path.exists(data_file):
    print(f'❌ Không tìm thấy {data_file}')
    print('   Hãy upload file asqp_annotated.jsonl vào data/processed/')
else:
    total = 0
    cat_counts = Counter()
    sent_counts = Counter()
    with open(data_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            d = json.loads(line)
            total += 1
            for q in d.get('quadruples', []):
                cat_counts[q.get('aspect_category', 'UNK')] += 1
                sent_counts[q.get('sentiment', 'UNK')] += 1

    print(f'✅ File: {data_file}')
    print(f'   Reviews: {total:,}')
    print(f'   Sentiment: {dict(sent_counts)}')
    print(f'   Categories: {len(cat_counts)}')
    print(f'   Top 5: {dict(cat_counts.most_common(5))}')

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 5b: Sync data lên Drive (để không phải prepare lại)
# ══════════════════════════════════════════════════════
import shutil, os

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive')
DRIVE_PROJECT = f'{DRIVE_ROOT}/tiki_absa'
os.makedirs(f'{DRIVE_PROJECT}/data/training', exist_ok=True)

# Sync data/training lên Drive
shutil.copytree('data/training', f'{DRIVE_PROJECT}/data/training',
                dirs_exist_ok=True)
print(f'✅ data/training synced → {DRIVE_PROJECT}/data/training')

In [ ]:
import os, shutil, torch

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive')
DRIVE_PROJECT = f'{DRIVE_ROOT}/tiki_absa'
COLAB_PROJECT = '/content/tiki'
TOTAL_EPOCHS = 15
PATIENCE = 5
CHECKPOINT_DIR = 'checkpoints/phobert'
MODEL_DIR = 'models/phobert'
RESULT_PATH = 'results/phobert_results.json'
DRIVE_CKPT = f'{DRIVE_PROJECT}/checkpoints/phobert'
DRIVE_MODEL = f'{DRIVE_PROJECT}/models/phobert'
DRIVE_RESULTS = f'{DRIVE_PROJECT}/results'



script_path = f'{COLAB_PROJECT}/src/training/train_phobert.py'
if not os.path.exists(script_path):
    raise FileNotFoundError(
        f'Khong tim thay {script_path}. Hay chay CELL 2 de copy project tu Drive ve /content/tiki truoc.'
    )
os.chdir(COLAB_PROJECT)
print(f'Working directory: {os.getcwd()}')

def atomic_copy(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = f'{dst}.tmp'
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)


def sync_dir_to_drive(local_dir, drive_dir):
    if not os.path.exists(DRIVE_ROOT) or not os.path.exists(local_dir):
        return
    os.makedirs(drive_dir, exist_ok=True)
    for name in os.listdir(local_dir):
        if name.endswith('.tmp'):
            continue
        src = os.path.join(local_dir, name)
        dst = os.path.join(drive_dir, name)
        if os.path.isfile(src):
            atomic_copy(src, dst)


def clean_checkpoints(ckpt_dir, drive_ckpt_dir):
    """Kiem tra epoch_N.pt, xoa file corrupt o ca local lan Drive."""
    if not os.path.exists(ckpt_dir):
        return []
    pts = [f for f in os.listdir(ckpt_dir)
           if f.startswith('epoch_') and f.endswith('.pt')]
    good = []
    for fname in sorted(pts, reverse=True):
        local_path = os.path.join(ckpt_dir, fname)
        try:
            torch.load(local_path, map_location='cpu')
            good.append(fname)
        except Exception as e:
            os.remove(local_path)
            drive_path = os.path.join(drive_ckpt_dir, fname)
            if os.path.exists(drive_path):
                os.remove(drive_path)
            print(f'  [X] {fname} corrupt ({type(e).__name__}) - da xoa local + Drive')
    return good


def is_training_complete(ckpt_dir, total_epochs, patience):
    if not os.path.exists(ckpt_dir):
        return False, 0
    pts = [f for f in os.listdir(ckpt_dir)
           if f.startswith('epoch_') and f.endswith('.pt')]
    if not pts:
        return False, 0
    epochs = sorted([int(p.replace('epoch_','').replace('.pt','')) for p in pts])
    latest = epochs[-1]
    if latest >= total_epochs:
        return True, latest
    path = os.path.join(ckpt_dir, f'epoch_{latest}.pt')
    try:
        ckpt = torch.load(path, map_location='cpu')
        if ckpt.get('no_improve', 0) >= patience:
            return True, latest
    except Exception:
        pass
    return False, latest


# Buoc 1: Xoa checkpoint corrupt o local + Drive
good = clean_checkpoints(CHECKPOINT_DIR, DRIVE_CKPT)
if good:
    epochs_good = sorted([int(f.replace('epoch_','').replace('.pt','')) for f in good])
    print(f'[PhoBERT] Checkpoints hop le: epoch {epochs_good}')
else:
    print('[PhoBERT] Khong co checkpoint hop le')

# Buoc 2: Quyet dinh train hay bo qua
done, latest_ep = is_training_complete(CHECKPOINT_DIR, TOTAL_EPOCHS, PATIENCE)
best_model_path = f'{MODEL_DIR}/best_model.pt'

if done and os.path.exists(best_model_path):
    try:
        ckpt = torch.load(best_model_path, map_location='cpu')
        val_m = ckpt.get('val_metrics', {})
        print(f'Training PhoBERT HOAN THANH (epoch {ckpt.get("epoch", "?")}): '
              f'AD-F1={val_m.get("ad_f1", 0):.4f} | AP-F1={val_m.get("ap_f1", 0):.4f}')
        print('Khong train lai. Muon train lai thi xoa checkpoints/phobert/ va models/phobert/.')
        sync_dir_to_drive(MODEL_DIR, DRIVE_MODEL)
        sync_dir_to_drive(CHECKPOINT_DIR, DRIVE_CKPT)
        if os.path.exists(RESULT_PATH):
            atomic_copy(RESULT_PATH, f'{DRIVE_RESULTS}/phobert_results.json')
    except Exception:
        done = False

if not done:
    if latest_ep > 0:
        print(f'Resume PhoBERT tu epoch {latest_ep + 1}/{TOTAL_EPOCHS}')
    else:
        print('Bat dau train PhoBERT tu epoch 1...')
    get_ipython().system(f'python {script_path}')